# Court Keypoint Identification Audit (Stage 7)

**Purpose:** An interactive audit collecting one human verdict per visible keypoint: is
index k on the landmark that index means? It scores the identification design registered
as section 4.4 of `docs/prereg/court_keypoint_spec.md`.  
**Inputs:** `data/raw/clip_*.mp4` and `models/keypoints.pt`, run through the production
detection and cache path.  
**Outputs:** `data/annotations/keypoint_audit.csv`, appended after every verdict,
resumable.  
**Backs:** `data/annotations/keypoint_audit.csv` and the identification-accuracy figures
reported from it.

This is the only Stage 7 measurement needing a human. Reprojection residuals check
whether keypoints are geometrically consistent with each other, which a systematic
left-right index swap satisfies perfectly by mapping to the mirrored court. That
proxy is blind to exactly the failure a near-symmetric court invites, and blind in
the direction of a falsely clean result.

The UI is matplotlib inline plus `input()`, deliberately not ipywidgets:
`jupyterlab_widgets` is not registered with this project's JupyterHub server process
and cannot be fixed from the user's account, so widgets do not render there.

The first cell pins the working directory to the repo root and imports the audit logic
from `basketball/labelling/keypoint_audit.py`, where it is unit-tested.

In [1]:
%matplotlib inline
import os
from pathlib import Path

# Every path below is relative to the repo root, not wherever Jupyter
# happened to start. Without this, data/raw/clip_1.mp4 resolves against
# scripts/ and load_video raises OSError: Cannot open video.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

import cv2
import matplotlib.pyplot as plt

from basketball.annotators.keypoint_annotator import KeypointAnnotator
from basketball.keypoints.court_template import KEYPOINT_NAMES
from basketball.labelling.keypoint_audit import (
    CLIPS,
    KEYPOINT_CONFIDENCE_THRESHOLD,
    build_verdict_order,
    eligible_frames,
    load_existing_verdicts,
    make_sampled_loader,
    run_session,
    sample_frames,
)

Working directory: /home/jovyan/nba-video-analytics


## 1. The index reference

Printed at the start of every session. Verdicting index 13 requires knowing what 13
is supposed to be, and expecting the labeller to hold 18 definitions in memory is a
source of error rather than a test of diligence.

In [2]:
for index in sorted(KEYPOINT_NAMES):
    print(f'{index:2d}  {KEYPOINT_NAMES[index]}')

 0  left baseline at sideline A
 1  left corner three-point, side A
 2  left lane corner at baseline, side A
 3  left lane corner at baseline, side B
 4  left corner three-point, side B
 5  left baseline at sideline B
 6  centre line at sideline B
 7  centre line at sideline A
 8  left free-throw line, side A
 9  left free-throw line, side B
10  right baseline at sideline B
11  right corner three-point, side B
12  right lane corner at baseline, side B
13  right lane corner at baseline, side A
14  right corner three-point, side A
15  right baseline at sideline A
16  right free-throw line, side A
17  right free-throw line, side B


## 2. Detection, sampling and session state

Runs the production keypoint detector over each clip through the cached path (the
fingerprint check regenerates a stale cache rather than silently reusing it), collects
each clip's eligible frames, and draws the audit sample reproducibly (seed 42, frames
presented in a fixed shuffle). The next cell reports how many verdicts exist already
and how many remain.

In [3]:
# Through run_detection with the production cache path, never load_cache:
# the fingerprint check must run, so a stale cache regenerates rather than
# being silently reused. These verdicts are the human ground truth for K3 and
# K5, and verdicts attached to a stale cache would be worse than no verdicts.
from basketball.keypoints.court_keypoints import CourtKeypoints
from basketball.labelling.keypoint_audit import (
    KEYPOINTS_CACHE_TEMPLATE,
    MODEL_PATH,
    RAW_VIDEO_TEMPLATE,
)
from basketball.utils.io_utils import load_video

detector = CourtKeypoints(
    MODEL_PATH, keypoint_confidence_threshold=KEYPOINT_CONFIDENCE_THRESHOLD,
)
keypoints_by_clip = {}
for clip in CLIPS:
    video_path = RAW_VIDEO_TEMPLATE.format(clip=clip)
    keypoints_by_clip[clip] = detector.run_detection(
        frames=[frame for _, frame in load_video(video_path)],
        video_path=video_path,
        cache_path=KEYPOINTS_CACHE_TEMPLATE.format(clip=clip),
    )
eligible_by_clip = {clip: eligible_frames(keypoints_by_clip[clip]) for clip in CLIPS}
sampled_by_clip = sample_frames(eligible_by_clip)

for clip in CLIPS:
    print(f'{clip}: {len(eligible_by_clip[clip])} eligible frames, '
          f'sampled {len(sampled_by_clip[clip])}: {sampled_by_clip[clip]}')

clip_loader = make_sampled_loader(sampled_by_clip)

clip_1: 117 eligible frames, sampled 8: [8, 10, 17, 28, 52, 59, 74, 77]
clip_2: 174 eligible frames, sampled 8: [4, 31, 39, 63, 116, 128, 135, 148]
clip_3: 243 eligible frames, sampled 9: [20, 25, 26, 103, 116, 132, 143, 183, 188]


In [4]:
verdict_order = build_verdict_order(sampled_by_clip, keypoints_by_clip)
done = load_existing_verdicts()

print(f'{len(verdict_order)} keypoint verdicts across '
      f'{len(set((c, f) for c, f, _ in verdict_order))} frames')
print(f'{len(done)} already recorded, {len(verdict_order) - len(done)} remaining')

148 keypoint verdicts across 25 frames
148 already recorded, 0 remaining


## 3. The labelling session

Each frame is rendered once, large, with every confident keypoint drawn as a dot
labelled by its index; you then verdict each index in turn while the image stays on
screen. A `w` (wrong landmark) verdict asks a follow-up for which index it actually
sits on, which is what scores prediction K5.

`s` stops. Progress is saved after every single verdict, so stopping is safe and
re-running this cell resumes where you left off.

In [5]:
# Rendering constants for the point under judgement. Deliberately distinct
# from KeypointAnnotator's yellow so the prompted index cannot be mistaken for
# its neighbours, which stay in the existing style because judging whether an
# index sits on the right landmark depends on seeing the others for context.
JUDGED_COLOUR = (0, 0, 255)      # BGR: red
JUDGED_RADIUS = 10
JUDGED_THICKNESS = 3
JUDGED_FONT_SCALE = 1.2
INSET_HALF_SIZE = 100            # a ~200x200 px crop around the judged point


def draw_judged(frame, keypoint):
    """Return a copy of an annotated frame with one keypoint redrawn as the point under judgement."""
    marked = frame.copy()
    x, y = int(round(keypoint.x)), int(round(keypoint.y))
    cv2.circle(marked, (x, y), JUDGED_RADIUS, JUDGED_COLOUR, thickness=JUDGED_THICKNESS)
    cv2.putText(
        marked, str(keypoint.index), (x + JUDGED_RADIUS + 4, y - JUDGED_RADIUS - 4),
        cv2.FONT_HERSHEY_SIMPLEX, JUDGED_FONT_SCALE, JUDGED_COLOUR, 2,
    )
    return marked


def inset_crop(frame, keypoint):
    """Return a crop centred on one keypoint, clamped so a point near an edge still yields a full-size window."""
    height, width = frame.shape[:2]
    x, y = int(round(keypoint.x)), int(round(keypoint.y))
    # Clamped rather than allowed out of bounds: a keypoint at the frame edge
    # would otherwise produce an empty or truncated crop, and edge keypoints
    # are exactly the ones prediction K3 expects to localise worst.
    x1 = max(0, min(x - INSET_HALF_SIZE, width - 2 * INSET_HALF_SIZE))
    y1 = max(0, min(y - INSET_HALF_SIZE, height - 2 * INSET_HALF_SIZE))
    x1, y1 = max(0, x1), max(0, y1)
    return frame[y1:y1 + 2 * INSET_HALF_SIZE, x1:x1 + 2 * INSET_HALF_SIZE]


def show(clip: str, frame_idx: int, keypoint_index: int) -> None:
    """Render one sampled frame with every confident keypoint labelled, the judged one highlighted, beside a zoomed crop of it."""
    data = clip_loader(clip)
    frame = data.frames[frame_idx]
    frame_keypoints = data.keypoints[frame_idx]
    annotated = KeypointAnnotator(
        render_threshold=KEYPOINT_CONFIDENCE_THRESHOLD,
    ).draw([frame], [frame_keypoints])[0]

    judged = next(kp for kp in frame_keypoints if kp.index == keypoint_index)
    annotated = draw_judged(annotated, judged)

    # Full frame large enough to judge context, inset large enough to read a
    # few pixels of displacement: the errors of interest are tens of pixels on
    # a 1280-wide frame, and the keypoint that motivated this audit was off by
    # only a few.
    figure, (full_axis, inset_axis) = plt.subplots(1, 2, figsize=(24, 9), width_ratios=[2, 1])
    full_axis.imshow(annotated[:, :, ::-1])
    full_axis.axis('off')
    full_axis.set_title(f'{clip} frame {frame_idx} - judging keypoint {keypoint_index}')

    inset_axis.imshow(inset_crop(annotated, judged)[:, :, ::-1])
    inset_axis.axis('off')
    inset_axis.set_title(f'{keypoint_index}: {KEYPOINT_NAMES[keypoint_index]}')
    plt.tight_layout()
    plt.show()


written = run_session(
    verdict_order=verdict_order,
    keypoints_by_clip=keypoints_by_clip,
    show=show,
    prompt=input,
)
print(f'{written} verdict(s) written this session.')

All 148 keypoints are verdicted — nothing left to do.
0 verdict(s) written this session.


## 4. Verify

Re-reads the CSV through the same last-wins dedup every consumer uses, so this
reports what scoring will actually see.

In [6]:
from collections import Counter

from basketball.labelling.keypoint_audit import load_labelled_rows

rows = load_labelled_rows()
print(f'{len(rows)} verdicts recorded (post-dedup)')
print(Counter(row['verdict'] for row in rows))

wrong = [row for row in rows if row['verdict'] == 'wrong_landmark']
print(f'{len(wrong)} wrong-landmark verdicts, actual indices: '
      f'{Counter(row["actual_index"] for row in wrong)}')

148 verdicts recorded (post-dedup)
Counter({'correct': 148})
0 wrong-landmark verdicts, actual indices: Counter()


## 5. Outcome

Verdicts accumulate in `data/annotations/keypoint_audit.csv`, one row per (clip, frame,
keypoint index), deduplicated last-wins on read. The shipped CSV is the human ground
truth behind the identification-accuracy figures and the scoring of predictions K3
and K5.